# Kinematic quantification and plotting

These demos target **movement 0.17.0** and one explicitly selected session.
`data-conduit` selects, reads and aligns the experiment's DLC and event streams;
`movement` filters the pose and calculates the measurements. Matplotlib arranges
panels and the shared template adds event labels.

The time axis remains the recorded session clock in **seconds**. Spatial values
remain **pixels**; no arena calibration is inferred. Filtering settings are
examples to review for this session, not validated paper analysis settings.
Processing runs on the continuous session before any trial is sliced.
No real-data output has been generated in this template.

In [ ]:
from pathlib import Path
import sys

# Locate this checkout when Jupyter starts in a notebook subdirectory.
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "movement_figures").is_dir():
        sys.path.insert(0, str(parent))
        break
    if (parent / "src" / "movement_figures").is_dir():
        sys.path.insert(0, str(parent / "src"))
        break
else:
    raise RuntimeError("Start Jupyter inside the data-conduit checkout.")

import matplotlib.pyplot as plt
import movement
import movement.kinematics as kin
import numpy as np
import pandas as pd
from IPython.display import display
from movement.plots import plot_centroid_trajectory, plot_occupancy
from movement.utils.vector import compute_signed_angle_2d

from movement_figures.data_template.loading import (
    build_config, load_session, prepare_pose, preview_session,
)
from movement_figures.timeseries_template.annotations import annotate_qc_timeseries, qc_annotations

print(f"movement {movement.__version__}: {movement.__file__}")

## Session and processing settings

In [ ]:
# Edit this cell for your data. The same settings apply to all three notebooks.
ROOT = Path("/path/to/Training")
SESSION = "replace-with-one-session-folder-name"
LEVEL_NAMES = ("mouseID", "day")  # ROOT / mouseID / day / session
LEVEL_SELECTORS = None  # e.g. {"l0_selector": ["your-mouse"]}
SOURCES = ("trials", "events", "dlc")

INDIVIDUAL = "individual_0"
TRACKING_KEYPOINT = "body"  # one tracked point, used consistently for paths/speed/occupancy
LEFT_EAR, RIGHT_EAR = "lear", "rear"
BODY_FRONT, BODY_BACK = "body", "tailbase"
CAMERA_VIEW = "top_down"  # DLC image coordinates: +x right, +y down
REFERENCE_VECTOR = (1.0, 0.0)  # fixed axis; register to arena axes if different

CONFIDENCE_THRESHOLD = 0.9  # example setting: inspect your own confidence distribution
MAX_GAP_FRAMES = None  # None disables interpolation; an integer fills only short gaps
SMOOTHING_WINDOW = None  # None disables movement's rolling median filter

TRIAL_ROWS = (0, 1, 2)  # zero-based rows of the displayed, time-sorted session trial table
WINDOW = None  # absolute seconds (start, end); None shows the first 60 seconds of trials
OCCUPANCY_BINS = 40
ARENA_RANGE = None  # optionally ((xmin, xmax), (ymin, ymax)), in original pixels

config = build_config(
    ROOT, session=SESSION, level_names=LEVEL_NAMES,
    level_selectors=LEVEL_SELECTORS, sources=SOURCES,
)

## Load, inspect coverage, and process

In [ ]:
# Preview rejects zero or multiple selected sessions before any source data is loaded.
display(preview_session(config))
session_data = load_session(config, individual=INDIVIDUAL)
display(session_data.loaded.stream_coverage)
raw_pose = session_data.raw_pose
pose = prepare_pose(
    raw_pose, confidence_threshold=CONFIDENCE_THRESHOLD,
    max_gap_frames=MAX_GAP_FRAMES, smoothing_window=SMOOTHING_WINDOW,
)
position = pose.position.sel(individual=INDIVIDUAL, drop=True)
if TRACKING_KEYPOINT not in position.keypoint:
    raise ValueError(f"Choose TRACKING_KEYPOINT from {position.keypoint.values.tolist()}")
point = position.sel(keypoint=TRACKING_KEYPOINT, drop=True)
point.attrs["units"] = "px"
trials = session_data.trials.sort_values("start_time").reset_index(drop=True)
events = session_data.events
if trials.empty:
    raise ValueError("No parsed trials are available in this session.")
display(trials)
for note in qc_annotations(trials, events).notes:
    print(note)
print("Keypoints:", position.keypoint.values.tolist())
print("Valid tracked-point frames:", int(point.notnull().all("space").sum()),
      "/", point.sizes["time"])

if WINDOW is None:
    window_start = max(float(point.time.min()), float(trials.start_time.iloc[0]))
    WINDOW = (window_start, min(window_start + 60, float(point.time.max())))
if not np.isfinite(WINDOW).all() or WINDOW[0] >= WINDOW[1]:
    raise ValueError("WINDOW must be a finite increasing interval overlapping the pose.")
if not bool(((point.time >= WINDOW[0]) & (point.time <= WINDOW[1])).any()):
    raise ValueError("WINDOW contains no recorded pose samples; choose a window on the session clock.")
selected_rows = [i for i in TRIAL_ROWS if 0 <= i < len(trials)]
if len(selected_rows) != len(TRIAL_ROWS):
    print("Trial rows outside this session were omitted:", sorted(set(TRIAL_ROWS) - set(selected_rows)))
if not selected_rows:
    raise ValueError("TRIAL_ROWS must select at least one displayed trial row.")

## Location, linear speed, and angular head velocity

Linear speed uses `movement.kinematics.compute_speed` on the selected tracked point.
Head direction comes from the forward vector perpendicular to the left/right ear
axis. Check that the ear labels and camera view match the recording.

`movement` has no dedicated angular-head-velocity function in 0.17.0. We compose
its head-direction and signed-angle functions, then divide each signed angular
change by its recorded frame interval. This estimates the average angular
velocity over the **preceding** interval. It handles the ±π branch cut without
differentiating wrapped headings. It assumes less than π rotation between
successive frames; the first sample and intervals touching missing head positions
remain NaN. In image coordinates (+y down), positive rotation is clockwise.

In [ ]:
speed = kin.compute_speed(point)
head = kin.compute_head_direction_vector(
    position, left_keypoint=LEFT_EAR, right_keypoint=RIGHT_EAR,
    camera_view=CAMERA_VIEW,
)
head_rotation = compute_signed_angle_2d(head.shift(time=1), head)
elapsed_seconds = head.time - head.time.shift(time=1)
angular_head_velocity = (head_rotation / elapsed_seconds).rename("angular_head_velocity")
angular_head_velocity.attrs["units"] = "rad/s"

In [ ]:
traces = [
    (point.sel(space="x"), "x location (px)"),
    (point.sel(space="y"), "y location (px)"),
    (speed, "Linear speed (px/s)"),
    (angular_head_velocity, "Angular head velocity (rad/s)"),
]
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True, layout="constrained")
for index, (ax, (trace, label)) in enumerate(zip(axes, traces)):
    visible = trace.sel(time=slice(*WINDOW))
    ax.plot(visible.time, visible, color="#5b4b9a", lw=0.9)
    ax.set_ylabel(label)
    if index == 2:
        ax.set_ylim(bottom=0)
    elif index == 3:
        limit = float(np.abs(visible).max(skipna=True))
        limit = limit if np.isfinite(limit) and limit > 0 else 1.0
        ax.set_ylim(-1.05 * limit, 1.05 * limit)
    annotate_qc_timeseries(ax, trials, events, window=WINDOW, legend=index == 0)
axes[-1].set_xlabel("Session time (s)")
fig.suptitle(f"{SESSION} — {TRACKING_KEYPOINT} location/speed and ear-based head rotation")
plt.show()

## The same measurements for individual trials

Columns show the selected rows, each aligned to its derived trial start. The raw
`Start Trial` event remains a separate marker. The measurements above are sliced
here; derivatives are not recalculated at each trial boundary.

In [ ]:
fig, axes = plt.subplots(2, len(selected_rows), squeeze=False, sharey="row",
                         figsize=(5 * len(selected_rows), 6), layout="constrained")
for column, trial_row in enumerate(selected_rows):
    trial = trials.iloc[trial_row]
    start, end = float(trial.start_time), float(trial.end_time)
    for row, (trace, label) in enumerate([(speed, "Linear speed (px/s)"),
                                         (angular_head_velocity, "Head velocity (rad/s)")]):
        ax = axes[row, column]
        visible = trace.sel(time=slice(start, end))
        ax.plot(visible.time - start, visible, lw=0.9, color="#5b4b9a")
        annotate_qc_timeseries(ax, trials.iloc[[trial_row]], events,
                              window=(start, end), time_offset=start,
                              legend=row == 0 and column == 0)
        ax.set_ylabel(label)
        if row == 0:
            ax.set_title(f"Trial row {trial_row}: {trial.outcome}")
        else:
            ax.set_xlabel("Time from derived trial start (s)")
axes[0, 0].set_ylim(bottom=0)
low, high = axes[1, 0].get_ylim()
limit = max(abs(low), abs(high))
axes[1, 0].set_ylim(-limit, limit)
plt.show()

API references: [speed](https://movement.neuroinformatics.dev/latest/api/movement.kinematics.compute_speed.html),
[head direction](https://movement.neuroinformatics.dev/latest/api/movement.kinematics.compute_head_direction_vector.html),
[signed angle](https://movement.neuroinformatics.dev/latest/api/movement.utils.vector.compute_signed_angle_2d.html).
Documentation may advance beyond the tested 0.17.0 version; record the version printed above.